[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/littlekg87/lecture/blob/main/2025_kmooc/notebooks/03_word_frequency.ipynb)


# 4-4차시 텍스트분석 실습 ① — 단어 빈도수 분석

『인조실록』에 실린 **김상헌의 상소문**을 단어 빈도수 분석으로 살펴봅니다.

**왼쪽의 ▶ 버튼을 위에서부터 차례대로 누르기만 하면 됩니다.**

> 💡 **왜 코랩을 쓰나요?**
> 영상에서는 터미널에 `pip install konlpy` 를 칩니다.
> 그런데 KoNLPy는 **자바(JDK)가 먼저 깔려 있어야** 해서, 특히 맥에서 애를 먹습니다.
> 코랩에서는 아래 준비 칸 한 번만 실행하면 끝납니다.


## 코드 설계

| 단계 | 하는 일 |
|---|---|
| **1단계** | 텍스트 가져오기 |
| **2단계** | 텍스트 전처리 — 불용어 제거, 형태소 분석 |
| **3단계** | 단어빈도수 계산 |
| **4단계** | 결과 출력 |

활용 라이브러리: **konlpy**


---
## 준비 — 라이브러리 설치

영상의 `pip install konlpy` 에 해당합니다. 자바까지 함께 깔아 줍니다. (1~2분)


In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
!pip install -q konlpy

import glob, os

# 설치된 자바 위치를 자동으로 찾아 알려줍니다
jvm = sorted(glob.glob('/usr/lib/jvm/java-*-openjdk-amd64'))
os.environ['JAVA_HOME'] = jvm[-1]

from konlpy.tag import Okt
print(Okt().nouns('설치가 잘 되었는지 확인합니다'))
print('설치 완료!')


## 준비 — 실습 데이터 내려받기


In [ ]:
# 실습 데이터를 강의 깃허브 저장소에서 곧바로 내려받습니다.
# 내 컴퓨터의 파일 경로를 적을 필요가 없습니다.

GITHUB = "https://raw.githubusercontent.com/littlekg87/lecture/main/2025_kmooc"

FILES = [
    "data/word-analysis/kim-sangheon-sangso.txt",
]

import os
import urllib.request

for path in FILES:
    name = path.split('/')[-1]
    try:
        urllib.request.urlretrieve(f'{GITHUB}/{path}', name)
    except Exception as e:
        raise SystemExit(
            f'내려받기에 실패했습니다: {path}\n'
            f'  · 인터넷 연결을 확인해 주세요.\n'
            f'  · 그래도 안 되면 강의 게시판에 문의해 주세요.\n'
            f'  (원인: {e})'
        )
    print(f'내려받음: {name}  ({os.path.getsize(name):,} 바이트)')

print()
print('준비 완료! 아래 칸부터 차례로 실행하세요.')

# (선택) 다른 텍스트로 해보고 싶다면 아래 두 줄의 # 을 지우고 실행하세요.
# from google.colab import files
# files.upload()


---
## 1단계 — 텍스트 가져오기


In [ ]:
import re
from collections import Counter
from konlpy.tag import Okt

# 파일 경로
file_path = 'kim-sangheon-sangso.txt'

# 텍스트 파일 읽기
with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

print(f'글자 수: {len(text):,}자')
print()
print(text[:300])  # 앞부분만 살짝 보기


> 📌 **강의 영상과 다른 점 — 파일 경로**
>
> 영상에서는 이 자리에 이런 긴 경로가 나옵니다.
> ```python
> file_path = r"C:\Users\littl\PycharmProjects\WordCloud_Practice\실습예제_김상헌상소문.txt"
> ```
> 이 경로는 **강사님 컴퓨터의 주소**라서 그대로 쓰면 오류가 납니다.
> 코랩에서는 파일을 방금 내려받아 바로 옆에 두었으므로,
> **파일 이름만 적으면 됩니다.**


---
## 2단계 — 텍스트 전처리 (불용어 제거, 형태소 분석)

- `[^가-힣\s]` 는 **한글과 공백이 아닌 모든 것**을 뜻합니다.
  한자·숫자·문장부호를 지워 분석 품질을 높입니다.
- `okt.nouns(text)` 는 문장에서 **명사만** 뽑아냅니다.


In [ ]:
# 한글 이외의 문자 제거
text = re.sub(r'[^가-힣\s]', '', text)

# 형태소 분석기 사용 (명사 추출)
okt = Okt()
tokens = okt.nouns(text)

print(f'추출된 명사: {len(tokens):,}개')
print(tokens[:30])


---
## 3단계 — 단어빈도수 계산


In [ ]:
# 단어 빈도수 계산
word_counts = Counter(tokens)

# 빈도수 기준 정렬
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)

print(f'총 {len(word_counts):,}종류의 단어')


---
## 4단계 — 결과 출력


In [ ]:
# 결과 출력
for word, freq in sorted_words:
    print(f'{word}, {freq}회')


> ✅ **제대로 됐는지 확인하기** — 맨 위가 이렇게 나와야 합니다.
>
> ```
> 것, 157회
> 수, 47회
> 말, 46회
> 그, 44회
> 일, 40회
> 신, 40회
> 사람, 39회
> ```
>
> 결과가 길면 출력 칸 왼쪽을 눌러 접을 수 있습니다.


> 🤔 **`것` 이 157회로 1등입니다. 이게 의미가 있을까요?**
>
> `것`, `수`, `그` 같은 단어는 **어느 글에나 많이 나오는 말**이라
> 이 상소문이 무엇에 관한 글인지 알려 주지 못합니다.
> 아래 **더 해보기** 에서 이런 단어를 걸러내 봅니다.


### 표로 정리하고 CSV로 내려받기


In [ ]:
import pandas as pd

df = pd.DataFrame(sorted_words, columns=['단어', '빈도'])
df.to_csv('단어_빈도_분석.csv', index=False, encoding='utf-8-sig')

display(df.head(20))

from google.colab import files
files.download('단어_빈도_분석.csv')


---
## 더 해보기 — 불용어(뜻 없는 단어) 걸러내기

> 여기부터는 **교안에 없는 내용**입니다. 결과가 영상과 달라집니다.

위 결과를 보면 `것`(157회), `수`, `그` 처럼 **뜻이 거의 없는 단어**가 상위를 차지합니다.
이런 단어를 **불용어(stopword)** 라고 합니다.

불용어를 걸러내면 **글의 내용을 실제로 보여 주는 단어**가 드러납니다.
무엇을 걸러낼지는 정답이 없고, **결과를 보고 목록을 고쳐 가며 반복**하는 것이 핵심입니다.


In [ ]:
# 걸러낼 단어 목록 — 결과를 보고 자유롭게 더하거나 빼세요
stopwords = ['것', '저', '그', '이', '수', '때', '등', '바', '더']

filtered = [w for w in tokens if w not in stopwords]

# 한 글자 명사는 뜻이 모호한 경우가 많아 함께 걸러 봅니다.
# 한 글자도 보고 싶다면 아래 줄 맨 앞에 # 을 붙이세요.
filtered = [w for w in filtered if len(w) > 1]

filtered_counts = Counter(filtered)

print('걸러내기 전 상위 10개')
for w, f in word_counts.most_common(10):
    print(f'  {w}, {f}회')

print()
print('걸러낸 뒤 상위 10개')
for w, f in filtered_counts.most_common(10):
    print(f'  {w}, {f}회')


---
### 정리

1. 단어빈도수 분석은 **텍스트 전처리 → 빈도수 분석** 의 순서로 코드를 작성한다.
2. 단어빈도수 분석에 어울리는 시각화를 할 수 있다.

다음 실습 → **[② 워드클라우드 시각화](04_wordcloud.ipynb)**
